# Lesson 1: Simple ReAct Agent from Scratch
# Lesson 2: Simple ReAct Agent with LangGraph

In [2]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [3]:
client = OpenAI()

In [4]:
chat_completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello world"}]
)

In [5]:
chat_completion.choices[0].message.content

'Hello! How can I assist you today?'

In [ ]:
# Agent parameterised by a system message
# Call method : take a message and append to the existing array of messages
# execute > call the API openai

class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

In [ ]:
# ReAct Agent need a specific message

prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [ ]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

# Toy to map the actions to the actual functions
known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [9]:
abot = Agent(prompt)

In [ ]:
result = abot("How much does a toy poodle weigh?")
print(result)
# there is a thought, an action and then a PAUSE

Thought: I should look up the average weight of a Toy Poodle using the average_dog_weight action.
Action: average_dog_weight: Toy Poodle
PAUSE


In [11]:
result = average_dog_weight("Toy Poodle")

In [12]:
result

'a toy poodles average weight is 7 lbs'

In [14]:
# We can then pass this to the next prompt
next_prompt = "Observation: {}".format(result)

In [15]:
abot(next_prompt)

'Answer: A Toy Poodle weighs an average of 7 lbs.'

In [ ]:
# look at what it does : messages of the agent
# system prompt, user, assistant, ... 
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

## Second part - more difficult question

In [19]:
# reinitiate the agent
abot = Agent(prompt)

In [20]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: I need to find the average weight of both a Border Collie and a Scottish Terrier, then add them together to find the combined weight.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [21]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [22]:
abot(next_prompt)

'Action: average_dog_weight: Scottish Terrier\nPAUSE'

In [23]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [24]:
abot(next_prompt)

'Action: calculate: 37 + 20\nPAUSE'

In [25]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [26]:
abot(next_prompt)

'Answer: The combined average weight of a Border Collie and a Scottish Terrier is 57 lbs.'

## Add loop

In [27]:
# regex to parse the LLM answer and decide is need to take an action
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

<>:2: SyntaxWarning: invalid escape sequence '\w'
<>:2: SyntaxWarning: invalid escape sequence '\w'
/var/folders/m6/v0tv2hpd70xbqzz_jv5c3m380000gn/T/ipykernel_44711/2024917649.py:2: SyntaxWarning: invalid escape sequence '\w'
  action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action


In [ ]:
def query(question, max_turns=5):
    i = 0                                   # keep track of iteration
    bot = Agent(prompt)                     # initialise the agent with default system prompt
    next_prompt = question
    while i < max_turns:                    # loop
        i += 1
        result = bot(next_prompt)           # result from the agent
        print(result)
        actions = [
            action_re.match(a)              # use regex to parse the actions
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:                         # if actions presents
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:                 # if not in the known actions - raise an error
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)       # look up in action dictionnary, then call that action
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)     # create the next prompt
        else:
            return

In [30]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together to find their combined weight.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: The combined average weight of a Border Collie and a Scottish Terrier is 57 lbs.


## Lesson 2: Simple ReAct Agent with LangGraph

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# different types of messages
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI                                     # langchain wraper
from langchain_community.tools.tavily_search import TavilySearchResults     # serach engine as tool

/Users/lyonrieublandera/Documents/Data_Science/Unige_course/mission_star_chat/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Need to add TAVILY_API_KEY to .env
tool = TavilySearchResults(max_results=4) #increased number of results
print(type(tool))
print(tool.name)

<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json


In [5]:
# create agent state : = annocated list of messages we add to over time
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [6]:
# creeate agent : need 3 functions (to use 2 as nodes and 1 as edge)
class Agent:

    # initiatlise with system message, tool, 
    def __init__(self, model, tools, system=""):
        self.system = system

        # initialise stategraog
        graph = StateGraph(AgentState)

        # create 3 functions
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}        # how to map the response
        )
        graph.add_edge("action", "llm")         # regular edge betwene action node and llm node
        graph.set_entry_point("llm")

        # will turn into a langchain runnable
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools} # map the name of the tool to the tool itself
        self.model = model.bind_tools(tools)    # pass the list of tools, tell the model it has thses available

    def exists_action(self, state: AgentState):
        """Function to define conditional edge if there is a message return True or """
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_openai(self, state: AgentState):
        """Function to add in system message, anreturn dictionnary """
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        """Function """
        tool_calls = state['messages'][-1].tool_calls
        results = []

        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        
        print("Back to the model!")
        return {'messages': results}

In [7]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOpenAI(model="gpt-3.5-turbo")  #reduce inference cost
abot = Agent(model, [tool], system=prompt)

In [9]:
from IPython.display import Image
#Image(abot.graph.get_graph().draw_png())

In [ ]:
# call cagent
messages = [HumanMessage(content="What is the weather in sf?")] # need to make it confrom to that state
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_BWmlPQ3om9u67w4L4moStJQu', 'type': 'tool_call'}
Back to the model!


In [ ]:
# is the final state the agent ended up in
result

{'messages': [HumanMessage(content='What is the weather in sf?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CbRnzx5EnGZd1SMC3vnZYgDKhiwXC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--c89322cd-1c5a-496a-bf0f-52affc67fc29-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_BWmlPQ3om9u67w4L4moStJQu', 'type': 'tool_call'}], usage_metadata={'input_tokens': 153, 'output_tokens': 21, 'total_tokens': 174, 'input_toke

In [ ]:
# get last message
result['messages'][-1].content

'The weather in San Francisco for November 2025 is expected to have moderate daytime temperatures of up to 16°C and cooler nights around 10°C. There will be moderate rainfall with a total of 66.8 mm spread over approximately 14 days.'

In [13]:
messages = [HumanMessage(content="What is the weather in SF and LA?")]
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_WtyN0xTSshNhG2QDct0NP3CU', 'type': 'tool_call'}
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Los Angeles'}, 'id': 'call_06i8JpKUJ7PXGhvuP0Rkr5XP', 'type': 'tool_call'}
Back to the model!


In [14]:
result['messages'][-1].content

'The current weather in San Francisco is cloudy with moderate rain, and temperatures ranging between 10°C and 18°C. In Los Angeles, the current weather is rainy with temperatures between 14°C and 24°C.'

In [15]:
# Note, the query was modified to produce more consistent results. 
# Results may vary per run and over time as search information and models change.

query = "Who won the super bowl in 2024? In what state is the winning team headquarters located? \
What is the GDP of that state? Answer each question." 
messages = [HumanMessage(content=query)]

model = ChatOpenAI(model="gpt-4o")  # requires more advanced model
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'tavily_search_results_json', 'args': {'query': '2024 Super Bowl winner'}, 'id': 'call_CA4eUSkrDVjLMIzbYuFXke3v', 'type': 'tool_call'}
Back to the model!
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'Kansas City Chiefs headquarters state'}, 'id': 'call_QHdGJJguKkLFvrxBPklzqJFg', 'type': 'tool_call'}
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'Missouri GDP 2024'}, 'id': 'call_9nzvPdKd2y9ul8apySHxyTYp', 'type': 'tool_call'}
Back to the model!


In [16]:
print(result['messages'][-1].content)

1. The Kansas City Chiefs won the Super Bowl in 2024.

2. The Kansas City Chiefs are headquartered in Kansas City, Missouri.

3. The real GDP of Missouri in 2024 was $344.12 billion, according to the data provided by USAFacts.
